In [24]:
# Installation automatique des dépendances requises dans le noyau Jupyter actuel
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


# 🧹 Étape 2 : Préparation & Nettoyage de Données (Data Wrangling) (Squelette Étudiant)

Cette étape correspond au deuxième chapitre du projet. L'objectif est d'effectuer un audit de qualité de vos données brutes, puis de mettre en œuvre un nettoyage rigoureux à l'aide de votre package personnalisé `src.data_clean`.

### 1. Initialisation et imports

In [11]:
import os
import sys
import pandas as pd
import numpy as np

BASE_DIR = os.path.abspath("..")
sys.path.append(BASE_DIR)

from src import data_clean as dc

print("🧹 Wrangling démarré")

🧹 Wrangling démarré


### 2. Chargement du dataset brut et Audit Initial

**À COMPLÉTER PAR L'ÉTUDIANT :**
Chargez les données brutes et inspectez la qualité du dataset (taux de valeurs manquantes, présence de doublons, types erronés).

In [12]:
raw_data_path = os.path.join(BASE_DIR, "data", "processed", "raw_merged.csv")

df_raw = pd.read_csv(raw_data_path)

print("✔ Dataset chargé :", df_raw.shape)

df_raw.info()

✔ Dataset chargé : (1460, 83)
<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 83 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Id                   1460 non-null   int64  
 1   MSSubClass           1460 non-null   int64  
 2   MSZoning             1460 non-null   str    
 3   LotFrontage          1201 non-null   float64
 4   LotArea              1460 non-null   int64  
 5   Street               1460 non-null   str    
 6   Alley                91 non-null     str    
 7   LotShape             1460 non-null   str    
 8   LandContour          1460 non-null   str    
 9   Utilities            1460 non-null   str    
 10  LotConfig            1460 non-null   str    
 11  LandSlope            1460 non-null   str    
 12  Neighborhood         1460 non-null   str    
 13  Condition1           1460 non-null   str    
 14  Condition2           1460 non-null   str    
 15  BldgType           

### 3. Uniformisation des Formats de Dates

**À COMPLÉTER PAR L'ÉTUDIANT :**
Uniformisez la colonne temporelle pour la convertir dans un type datetime standardisé via votre module `src.data_clean`.

In [13]:
##Missing values intelligents
# numériques
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/data_merged.csv")  # adapte si besoin

df.head()
num_cols = df.select_dtypes(include=[np.number]).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# catégorielles
cat_cols = df.select_dtypes(include=["object"]).columns

for col in cat_cols:
    df[col] = df[col].fillna("Unknown")

print("✔ Missing values traitées")

print("\n❌ Valeurs manquantes (top 15)")
print(df_raw.isnull().sum().sort_values(ascending=False).head(15))

print("\n🔁 Doublons :", df_raw.duplicated().sum())

df_raw["zone_type"] = df_raw["zone_type"].fillna("Unknown")
df_raw["coef_multiplicateur"] = df_raw["coef_multiplicateur"].fillna(1.0)

if "LotFrontage" in df_raw.columns:
    df_raw["LotFrontage"] = df_raw["LotFrontage"].fillna(df_raw["LotFrontage"].median())



for col in ["GarageCars", "GarageArea", "TotalBsmtSF"]:
    if col in df_raw.columns:
        df_raw[col] = df_raw[col].fillna(0)

✔ Missing values traitées

❌ Valeurs manquantes (top 15)
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageFinish      81
GarageCond        81
GarageYrBlt       81
GarageQual        81
GarageType        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
dtype: int64

🔁 Doublons : 0


/var/folders/q4/93wvzxx506dbz8ycqp90l57m0000gn/T/ipykernel_70153/142319559.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object"]).columns


### 4. Identification et Filtrage des Valeurs Aberrantes (Outliers)

**À COMPLÉTER PAR L'ÉTUDIANT :**
Identifiez les anomalies physiques et utilisez votre fonction `dc.handle_outliers` pour transformer ces valeurs aberrantes en NaNs.

In [14]:
def remove_outliers(df, col):
    q1 = df[col].quantile(0.01)
    q99 = df[col].quantile(0.99)
    df[col] = df[col].clip(q1, q99)
    return df

if "SalePrice" in df_raw.columns:
    df_raw = remove_outliers(df_raw, "SalePrice")
    print("✔ Outliers SalePrice traités")

✔ Outliers SalePrice traités


### 5. Imputation des valeurs manquantes

**À COMPLÉTER PAR L'ÉTUDIANT :**
Appliquez des stratégies d'imputation adaptées (interpolation temporelle, médiane, etc.) sur les valeurs manquantes générées ou initiales.

In [15]:
if "YearBuilt" in df_raw.columns:
    df_raw["HouseAge"] = 2026 - df_raw["YearBuilt"]

if "GrLivArea" in df_raw.columns:
    df_raw["LogArea"] = np.log1p(df_raw["GrLivArea"])

print("\n❌ Missing après nettoyage")
print(df_raw.isnull().sum().sort_values(ascending=False).head(10))

df_final = df_raw.copy()


❌ Missing après nettoyage
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
dtype: int64


### 6. Sauvegarde des données propres

Enregistrez vos données de base nettoyées dans le répertoire `data/processed/`.

In [16]:
processed_path = os.path.join(BASE_DIR, "data", "processed", "cleaned_data_sample.csv")

os.makedirs(os.path.dirname(processed_path), exist_ok=True)

df_final.to_csv(processed_path, index=False)

print("💾 Données propres sauvegardées :", processed_path)
print("🚀 Wrangling terminé avec succès")

💾 Données propres sauvegardées : /Users/hobb/Documents/GitHub/aptispace-datascience-projet/data/processed/cleaned_data_sample.csv
🚀 Wrangling terminé avec succès
